In [9]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
from enum import Enum
from typing import Literal
from dotenv import load_dotenv

##load key
load_dotenv()

##Sample Prompts
sample_message = """
Hi team, I was charged twice for my subscription this month! 
My invoice shows two $49.99 charges on my credit card statement for order #98452. 
I need a refund for the extra charge ASAP because it's messing up my budget. 
"""
sample_bug_message = """
Hey support, since the latest iOS update this morning, the app crashes instantly 
every time I tap the 'Checkout' button on my cart screen. 
I'm on an iPhone 15 running iOS 18.2. I've already tried clearing my cache 
and reinstalling the app twice, but it keeps happening. I can't purchase anything!
"""

# ==========================================
# 1. DEFINE SCHEMA (Data Contract) . 

#Literal vs. Python Enum
#⚬	Literal (Inline Choices): You define the allowed values directly inside the type hint. It is fantastic for quick data validation and LLM structured outputs because LLM libraries (like Pydantic and Instructor) map Literal types directly to JSON Schema enum constraints.
#⚬	Enum (Class-based): You create a formal class inheriting from enum.Enum. This is better if you have a massive list of options, need to reuse them across multiple files, or want to attach methods and properties to the choices.
# ==========================================
class PriorityEnum(str,Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    URGENT = "Urgent"
    
class IssueClassifier(BaseModel):
    category: Literal["Billing", "Technical", "Account", "General"] = Field(
        description="The primary category of the support request."
    )
    priority: PriorityEnum = Field(
        description="The urgency level."
    )
    summary: str = Field(description="A concise summary of the core problem.")

# ==========================================
# 2. SETUP THE OUTPUT PARSER
# =========================================

output_parser = PydanticOutputParser(pydantic_object=IssueClassifier)

# ==========================================
# 3. CREATE THE PROMPT TEMPLATE
# ==========================================
# We define placeholders for both the raw text (`{user_message}`) 
# and the parser instructions (`{format_instructions}`).

prompt_template = """
You are an intelligent customer support routing assistant. 
Analyze the incoming customer message and extract the following details precisely:

1. category: Select the best fit from [Billing, Technical, Account, General].
2. priority: Determine urgency from [Low, Medium, High, Urgent] based on business impact and customer sentiment.
3. summary: Write a crisp 1-2 sentence summary of the issue.

Customer Support Message:
\"\"\"
{user_message}
\"\"\"

{format_instructions}
"""
prompt=ChatPromptTemplate.from_template(prompt_template)

# ==========================================
# 4. PARTIAL THE PROMPT WITH INSTRUCTIONS
# ==========================================
support_prompt= prompt.partial(format_instructions=output_parser.get_format_instructions())
# Initialize the LLM model
llm_model = ChatAnthropic(temperature=0.0, model="claude-haiku-4-5-20251001")

# ==========================================
# 5. ASSEMBLE THE LCEL CHAIN
# ==========================================
# We link the components using the pipe (|) operator:
# Prompt Template -> LLM -> PydanticOutputParser -> Pydantic validation
#```
#raw LLM output
#      ↓
#parse
#      ↓
#structured Python/Pydantic object
#      ↓
#validation
#```
chain =  support_prompt | llm_model | output_parser

# ==========================================
# 6. EXECUTE / INVOKE THE CHAIN
# ==========================================
# We pass the required dictionary key ('text') matching our template variable.
# 1. Define your sample user message

result : IssueClassifier = chain.invoke({"user_message": sample_message})

# Output the strongly-typed Pydantic object
print(result)
print(f"Category : {result.category}")
print(f"Whats the priority: {result.priority}")
print(f"Summary of the issue: {result.summary}")

result2 : IssueClassifier = chain.invoke({"user_message": sample_bug_message})

# Output the strongly-typed Pydantic object
print(result2)
print(f"Category : {result2.category}")
print(f"Whats the priority: {result2.priority}")
print(f"Summary of the issue: {result2.summary}")

category='Billing' priority=<PriorityEnum.HIGH: 'High'> summary='Customer was charged twice for their subscription ($49.99 x 2) on order #98452 and requires an immediate refund for the duplicate charge.'
Category : Billing
Whats the priority: PriorityEnum.HIGH
Summary of the issue: Customer was charged twice for their subscription ($49.99 x 2) on order #98452 and requires an immediate refund for the duplicate charge.
category='Technical' priority=<PriorityEnum.URGENT: 'Urgent'> summary="App crashes immediately when tapping 'Checkout' button on iOS 18.2 after latest update, preventing purchases despite troubleshooting attempts."
Category : Technical
Whats the priority: PriorityEnum.URGENT
Summary of the issue: App crashes immediately when tapping 'Checkout' button on iOS 18.2 after latest update, preventing purchases despite troubleshooting attempts.
